# DWthon - Dzień 2: Zanurzamy się głębiej (Struktura i Kontrola)

Cześć. Witaj w drugim dniu.

Wczoraj zrobiliśmy solidny fundament. Wiesz już, dlaczego **Poziom 1 (Chat)** to ślepa uliczka dla biznesu i dlaczego musimy wejść na **Poziom 2 (Programowalne AI)** lub wyżej.

Dzisiaj więcej "pobrudzimy" sobie ręce w kodzie.


### Czego nauczysz się dzisiaj (dzień 2)

1. Zobaczysz na własne oczy, jak model zderza się z brudnymi danymi z prawdziwego świata i dlaczego prosty prompt to za mało, żeby okiełznać ludzką kreatywność w tytułach ogłoszeń
2. Nauczysz się pisać instrukcje dla modelu bezpośrednio w definicji typów danych, zamiast w oddzielnych blokach tekstu, co nazywamy podejściem *prompt as code*
3. Zrozumiesz, jak wymusić na modelu wybór jednej konkretnej wartości ze zdefiniowanej listy za pomocą typu Literal, co jest kluczem do standaryzacji danych w bazie
4. Przekonasz się, że błąd walidacji to nie porażka, ale informacja zwrotna i sukces architektury, która chroni Twój system przed wpuszczeniem śmieciowych danych
5. Poznasz mechanizm pętli naprawczej, dzięki któremu biblioteka instructor automatycznie pokazuje modelowi jego błąd i prosi o poprawkę bez Twojej ingerencji
6. Odkryjesz strategię bramkarza, czyli użycie walidatorów logicznych przed właściwym sprawdzeniem typu, aby naprawiać typowe błędy modelu zwykłym kodem Pythona
7. Zbudujesz hybrydowe rozwiązanie łączące nieokreśloność sztucznej inteligencji ze sztywną logiką inżynierską, co pozwoli Ci oszczędzać tokeny i zwiększyć stabilność systemu


Brzmi ciekawie? No to lecimy.

## Zagadka na dziś 

Teraz zrobię takie zadanie dla Ciebie.

Jest dzień i na tej scenie jest dużo zwierząt. Trzeba odpowiedzieć, co to za zwierzę.


<img src="../images/giraffe.jpg" />


Odpowiedź jest oczywista, ale ta sama zagadka jest dość trywialna — praktycznie od razu powiesz, co to jest. Swoją drogą, jaka jest odpowiedź?

Ale zobacz, co tu się stało. Przyjrzyjmy się temu, bo w tym momencie, przez to że finalny obrazek jest już prawie w 100% gotowy, w Twojej głowie nie może nagle pojawić się krokodyl, kot czy kapibara 🦫 ani żadne inne zwierzę.

Innymi słowy, kontekst — albo tory myślenia — zostały tak mocno narzucone, że Twój mózg jednoznacznie widzi tylko jedną możliwą ścieżkę.

Możemy pomyśleć, że to, co robimy ze strukturą dla LLM, działa bardzo podobnie. Chcemy dostarczyć mu na tyle precyzyjnych informacji, pewien "obraz", żeby nie miał zbyt dużej swobody wchodzenia na dziwne ścieżki, tylko dokładnie uzupełnił to, o co go prosimy.

## Wczytujemy biblioteki

Dość standardowy zestaw.

In [ ]:
from openai import OpenAI
from typing import List, Optional, Literal, Any
from pydantic import BaseModel, Field, model_validator
from dotenv import load_dotenv
import instructor
import pandas as pd
from tqdm import tqdm

## Inicjalizacja klienta

Tutaj bez zmian. Łączymy się z naszym Labem.
Kluczowe jest to: `instructor.patch(..., mode=instructor.Mode.MD_JSON)`.

To jest ten moment, w którym zakładamy modelowi "kaganiec". 
Od teraz nie będzie już radosnym poetą, ale będzie próbował zachowywać się jak funkcja w kodzie.

Przynajmniej powinno działać, a jeśli nie – złapiemy wyjątek (o tym szczegółowo później).

In [ ]:
load_dotenv()

In [ ]:
client = OpenAI(
    base_url = "https://llm-api.dataworkshop.eu"
)

client = instructor.patch(client, mode=instructor.Mode.MD_JSON)

## Funkcja pomocnicza `call_api`

Żeby nie pisać ciągle tego samego kodu, przygotowałem mały "wrapper".

Ta funkcja robi trzy rzeczy:
1.  Wysyła zapytanie do modelu
2.  Pilnuje, żeby odpowiedź pasowała do naszej klasy (`response_model`)
3.  Jeśli model się pomyli (zwróci zły format), `instructor` automatycznie poprosi go o poprawkę (`max_retries`). O tym jeszcze osobno wytłumaczę (zastosujmy dzisiaj).

To jest właśnie ta **automatyzacja kontroli jakości**, o której mówiłem wczoraj.

In [ ]:
def call_api(messages, response_model, model, max_retries=1):
    response = client.chat.completions.create(
        model    = model,
        messages = messages,
        temperature = 0,
        response_model = response_model,
        max_retries = max_retries
    )

    return response

## `temperature = 0`
Zwróć uwagę, że ustawiamy temperature = 0, ale... ten parametr ma zbyt dobrą reputację i zwykle kojarzy się tak: jak ustawisz go na zero, model zaczyna nas "słuchać" i przestaje halucynować.

<div class="alert alert-block alert-danger">

<p>To nieprawda! To kolejny mit.</p>
</div>


Owszem, zwykle ustawiamy go na mniejszą wartość (lub zero), ale to nadal zdecydowanie za mało, aby w pełni "opanować" model. Główna metoda, która przynajmniej jakoś próbuje zmotywować (zmusić) model do robienia tego, co trzeba, to głównie rozmowa przez odpowiednią strukturę promptu.

Dlatego zwykle robimy tak: ustawiamy `temperature = 0` lub małą wartość i… zapominamy. Wiemy, że to nie wystarczy, ale też nie ma co dalej tunować – mniej niż zero się nie da ustawić 🙂


## Szybki test (Sanity Check)

Zanim rzucimy się na głęboką wodę, sprawdźmy, czy wszystko działa ( z dnia 1).

Definiujemy prostą klasę `Response`. Zauważ:
-   `class`: To standardowa klasa Pythona.
-   `BaseModel`: To fundament z Pydantic.
-   `name: str`: To nasza instrukcja dla modelu -> "Oczekuję pola tekstowego".

In [ ]:
class Response(BaseModel):
    name: str

In [ ]:
call_api(
    [{"role": "user", "content": "Jak masz na imię?"}], 
    Response,
    "speakleash/Bielik-11B-v3.0-Instruct"
) 

## Czemu Bielik urkywa swoje imię?

To, co robiliśmy poprzedniego dnia, jak widzisz, wcale nie jest takie proste – nie da się łatwo "wydobyć" imienia od Bielika. Może to wynikać z kilku powodów, ale jest to dobry przykład sytuacji, w której próbujemy wyciągnąć z modelu wiedzę, której on po prostu nie posiada.

W takim przypadku model zaczyna zwracać cokolwiek, byle jakoś się dopasować (do naszych potrzeb). Może wtedy zachowywać się jak czysty generator tekstu. Inaczej jest w sytuacji, gdy dostarczymy fragment tekstu i powiemy: "Hej, znajdź w tym tekście np. imię albo coś innego." To jest już wyższy poziom użycia – wtedy LLM działa bardziej jako narzędzie do przetwarzania i wykrywania wzorców w dostarczonej treści, a nie jako baza wiedzy.

Na tym etapie najważniejsze jest to, że potrafimy zdefiniować strukturę, przekazać ją do modelu i poprawnie odebrać wynik. W kolejnym kroku, w naszym zadaniu, skupiamy się przede wszystkim na treści pochodzącej z ogłoszeń oraz na tym, aby wydobywać z niej właściwe informacje.

## Dane wejściowe, czyli realny świat

Wczoraj rzuciliśmy okiem na dane. Dzisiaj musimy z nimi pracować.
Wczytujemy 4000 ogłoszeń. Skupimy się na kolumnie `raw_title`.

Dlaczego `raw`? Bo to surowy tekst wpisany przez człowieka. A ludzie są... kreatywni.

In [ ]:
df = pd.read_parquet("../input/jobs.parquet")
df.sample(5)

### Problem i dekompozycja

Gdybyśmy chcieli wyciągnąć wszystko naraz (seniority, technologie, zarobki, pracę zdalną), model mógłby się pogubić (szczególnie model 11B).

Dlatego pragmatyczne podejście to **izolacja problemu**.
Na początek chcemy wyciągnąć z tytułu tylko jedną informację: **Jaki to poziom stanowiska (Seniority)?**

Zobaczmy przykłady, z czym będziemy walczyć. Wybrałem zestaw, który dobrze pokazuje bałagan w nazewnictwie:

In [ ]:
df["raw_title"].sample(50).values

### Skupmy się na poziomie zaawansowania (seniority)

Zamiast rzucać się na wszystko naraz, zróbmy to jak inżynierowie: izolacja (dekompozycja) problemu.
Na początek chcemy wyciągnąć z tytułu tylko jedną informację: **Jaki to poziom [zaawansowania] stanowiska?**

Wybrałem z bazy zestaw testowy, który dobrze pokazuje, z czym będziemy się mierzyć.
To nie są idealne przykłady z podręcznika. To jest "życie":

```text
- Staff Software Engineer - Big Data DevOps
- Senior Analytics Engineer (Multiple domains)
- Data Scientist, Algorithms - Pricing
- Senior Data Scientist - Remote - Latin America
- Senior Machine Learning Engineer
- Research Scientist / Ethics & Society
- Senior Research Engineer - Machine Learning (Machine Translation)
- Lead Data Engineer (REF964O) + sign on Bonus 5000 Eur
- Senior Business Intelligence Analyst -  Looker
- Senior Manager, Business Intelligence, InTech
- Jr. Pricing Analyst (Business Intelligence)
- [Job-11607] Mid-level Data Developer, Brasil
```

Spójrz na tę listę powyżej. Człowiek rozumie ją dość szybko, ale też szybko się męczy i wcale nie da się tego łatwo ani tanio skalować. Dla algorytmu (bez AI) to prawdziwy koszmar:

1.  **Różne nazewnictwo:** Mamy standardowe `Senior`, ale mamy też `Lead`, `Staff` (często mylone z "staffing agency"), czy `Manager`.
2.  **Skróty:** Zamiast "Junior" pojawia się `Jr.`.
3.  **Szum:** Kod referencyjny `(REF964O)`, kwota bonusu `5000 Eur`, albo ID w nawiasie kwadratowym `[Job-11607]`.
4.  **Lokalizacja i domena:** Wplecione w środek tekstu (`Remote`, `Latin America`, `Looker`).

Chcemy, aby na podstawie tego **Inputu** (tytuł), model zwrócił nam czysty **Output** – ustandaryzowany poziom (np. `Junior`, `Mid`, `Senior`, `Lead`, `Manager`).

Zacznijmy to robić krok po kroku. Metodą prób i błędów.

### Podejście 1: Naiwne (wartość może być dowolna - `str`, czyli tekst) - "baby step"

Zdefiniujmy pole `seniority_level` jako `Optional[str]`.
`Optional` oznacza, że jeśli model nie znajdzie informacji, może zwrócić `None` (zamiast zmyślać).

In [ ]:
class JobTitleV1(BaseModel):
    seniority_level: Optional[str]

call_api(
    [{"role": "user", "content": "Staff Software Engineer - Big Data DevOps"}], 
    JobTitleV1,
    "speakleash/Bielik-11B-v3.0-Instruct"
)    

### Wniosek z `JobTitleV1`
Model zwrócił: `'Staff Software Engineer - Big Data DevOps'`.
Czyli po prostu przepisał tytuł. To nam nic nie daje. Nadal mamy bałagan, tylko zamknięty w JSON-ie.

Musimy być bardziej precyzyjni. Robimy krok do przodu.

### Podejście 2: Dodajemy opis (prompt w kodzie lub to nazywa się "promt as code")

W **Software 3.0** nie piszemy instrukcji w `if-ach`, piszemy je w definicjach typów.
Użyjmy `Field(..., description="...")`. To jest miejsce, gdzie tłumaczymy modelowi, co ma zrobić z tym polem.

Instrukcja: *"Zostaw TYLKO doświadczenia wymagane na stanowisku"*.

#### 🤔 Możesz się zastanawiać, czy jest różnica, gdzie napiszemy tę instrukcję. Przecież moglibyśmy napisać to samo poza strukturą, prawda?

Możemy, ale różnica jest. Generalnie model (który wspiera w *Structure Output*) znacznie lepiej odnajduje się, kiedy ma konkretny wsad. Czasem dodatkowo powtarzamy instrukcje na poziomie "gołego" promptu, aby przypomnieć LLM, co ma zrobić. Natomiast możesz zapamiętać, że dodając `description="..."`, tekst trafia bezpośrednio do schematu i wtedy jest inaczej osadzony – staje się bardziej precyzyjny. Niby ta sama instrukcja, ale osadzona w schemacie daje zwykle znacznie lepszy wynik.

Na tym polega idea pisania promptów jako kodu (struktura), czyli inaczej niż w rozmowie między ludźmi. Z drugiej strony taki schemat pomaga zawęzić możliwe ścieżki, po których LLM może iść. Fachowo nazywa się to "minimalizowaniem entropii" – czyli zmniejszeniem liczby możliwych "dziwniejszych halucynacji", gdzie model mógłby pójść, gdyby nie miał wskazanych torów.

Swoją drogą, człowiek też często lepiej radzi sobie ze strukturą – choć tu temat jest bardziej skomplikowany niż w przypadku maszyn.

In [ ]:
class JobTitleV2(BaseModel):
    seniority_level: Optional[str] = Field(
        None, 
        description="Zostaw TYLKO doświadczenia wymagane na stanowisku. Tylko jedno słowa!"
    )   

call_api(
    [{"role": "user", "content": "Staff Software Engineer - Big Data DevOps"}], 
    JobTitleV2,
    "speakleash/Bielik-11B-v3.0-Instruct"
)     

### Wniosek z `JobTitleV2`
Jest lepiej. Model wyciągnął "Staff".
Ale spróbujmy czegoś trudniejszego.

Np. `"Jr. Pricing Analyst (Business Intelligence)"`

In [ ]:
 call_api(
    [{"role": "user", "content": "Jr. Pricing Analyst (Business Intelligence)"}], 
    JobTitleV2,
    "speakleash/Bielik-11B-v3.0-Instruct"
) 

Zwrócił `'Jr.'` lub `'Junior'`.

Problem polega na tym, że to wciąż jest `str` (tekst). Raz zwróci "Junior", raz "Jr.", raz "Entry-Level".
W bazie danych chcemy mieć porządek, a nie 50 określeń na to samo. Prawda?

### Lead Data Engineer (REF964O) + sign on Bonus 5000 Eur
Zaraz sobie coś wymyślmy, sprawdźmy jeszcze inny przypadek z `JobTitleV1` oraz `JobTitleV2`.

In [ ]:
call_api(
    [{"role": "user", "content": "Lead Data Engineer (REF964O) + sign on Bonus 5000 Eur"}], 
    JobTitleV1,
    "speakleash/Bielik-11B-v3.0-Instruct"
) 

In [ ]:
call_api(
    [{"role": "user", "content": "Lead Data Engineer (REF964O) + sign on Bonus 5000 Eur"}], 
    JobTitleV2,
    "speakleash/Bielik-11B-v3.0-Instruct"
) 

Jak uruchomisz kilka razy, zobaczysz, że wersja `JobTitleV1` jest bardziej chaotyczna. 
Natomiast `JobTitleV2` nadal zwraca za dużo, np. `Lead Data Engineer`, a chcemy, żeby było np. samo `Lead`.

### Podejście 3: 🦾 Żelazna ręka (`Literal`)

Tu wchodzi **`Literal`**.

To mechanizm, który mówi modelowi:
> *"Możesz wybrać wartość TYLKO z tej listy. Jeśli spróbujesz zwrócić cokolwiek innego – rzucę błędem."*

Definiujemy nasz (`Literal`). Lista możliwych wartości.

In [ ]:
SeniorityLevel = Literal[
    "Intern",       # Student, Stażysta
    "Junior",       # 0-2 lata
    "Mid-Level",    # 2-5 lat (lub brak info)
    "Senior",       # 5+ lat
    "Lead",         # Tech Lead / Team Lead
    "Staff",        # Staff Engineer
    "Principal",    # Principal / Distinguished
    "Manager",      # Engineering Manager
    "Director",     # Director / Head of
    "Executive",    # VP / C-Level
]


class JobTitleV3(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None, 
        description="Poziom doświadczenia wymagany na stanowisku"
    )

### Zwrócić uwagę, jak powstaje taka lista

W tej chwili wpisałem pewne wartości i wcale nie jest to trywialne ani łatwe. To jest coś, co trzeba "rzeźbić", bo pojawia się kilka wyzwań.

Po pierwsze, wprowadzamy pewne wartości, np. `Senior`, `Lead` albo `Staff`. W teorii każdy z tych tytułów ma swoje uzasadnienie, ale problem polega na tym, że nawet w obrębie jednej firmy trudno jednoznacznie odróżnić, gdzie kończy się `Senior`, gdzie zaczyna `Lead`, a gdzie `Staff` lub `Principal`.

Teraz pomyślmy o tytułach z różnych firm – tu sytuacja jest jeszcze trudniejsza. Nie mamy do czynienia tylko ze stanem faktycznym, ale bardziej z dopasowaniem do treści tekstu. Dlatego pojawia się tu pewne wyzwanie. Na ten moment założyłem pewne rzeczy, analizując je manualnie lub mając pełną wiedzę. Jednak problemy, które mogą się pojawić, wymagają czasu, żeby je przemyśleć i uporządkować w odpowiedni wymiar.

Zwróćmy też uwagę na przypadki mieszane, np. `Senior Manager`. Trzeba wtedy zdecydować: `Senior` czy `Manager`? I to już nie jest takie proste. Ten przypadek obsłużymy później trochę inaczej, prowadząc kilka kolejnych wymiarów. Na razie patrzymy na to "płasko" – niezależnie od kombinacji, musi być wybrana jedna opcja.

Mówię o tym przede wszystkim po to, żeby pokazać, że gdy zaczynasz myśleć poprzez schemat, rozumiesz, że nakładanie struktury wymaga wysiłku. Trzeba przemyśleć, jak najlepiej ją zorganizować.


### Test masowy

Przygotowałem listę 3 różnych klas (`V1`, `V2`, `V3`) i zestaw trudnych przykładów.
Zobaczmy, jak radzą sobie poszczególne wersje.

In [ ]:
so_list = [
    JobTitleV1,
    JobTitleV2,
    JobTitleV3
]

samples = [
    "Staff Software Engineer - Big Data DevOps",
    "Senior Analytics Engineer (Multiple domains)",
    "Data Scientist, Algorithms - Pricing",
    "Senior Data Scientist - Remote - Latin America",
    "Jr. Pricing Analyst (Business Intelligence)",
    "[Job-11607] Mid-level Data Developer, Brasil",
]


results = {}

for so in tqdm(so_list):
    so_str = str(so.__name__)
    if so_str not in results:
        results[so_str] = []
    
    for sample in samples:
        response = call_api(
            [{"role": "user", "content": sample}], 
            so,
            "speakleash/Bielik-11B-v3.0-Instruct"
        ) 

        results[so_str].append(response.seniority_level)

df_results = pd.DataFrame(results, index=samples)
df_results

### Analiza wyników

Widzisz różnicę?
-   `JobTitleV1`: Chaos (całe zdania)
-   `JobTitleV2`: Lepiej, ale niespójnie (raz "Mid-level", raz "Mid").
-   `JobTitleV3`: Czysta struktura. "Jr." zamieniło się w "Junior". "Mid-level" jest ustandaryzowane.

To jest siła napędowa jak kontrolujemy rzeczy w **Software 3.0**. Przesunęliśmy odpowiedzialność za "rozumienie" na model, ale odpowiedzialność za "standard" wzięliśmy na siebie (poprzez `Literal`).

Ale… zaraz coś się zepsuje (żeby potem to naprawić). Kod poniżej rzuca wyjątek – nie przestrasz się 💥

### `Literal` ma być jednoznaczny (i to na liczymy)

Spróbujmy wrzucić tytuł: `"Senior Manager, Business Intelligence"`.
Zgodnie z naszą logiką, powinno to być "Manager" albo "Senior". Zobaczmy, co zrobi `JobTitleV3`.

In [ ]:
call_api(
    [{"role": "user", "content": "Senior Manager, Business Intelligence, InTech"}], 
    JobTitleV3,
    "speakleash/Bielik-11B-v3.0-Instruct"
) 

### `ValidationError` – Dlaczego to jest dobre?

Masz ścianę czerwonego tekstu? **Świetnie.** Brzmi dziwnie?
To nie jest porażka, to jest sukces architektury. Mały powód dla dumy – LLM chciał zrobić po swojemu, a Ty masz bramkę jakości i teraz możesz coś z tym zrobić :)

Zobacz błąd (najważniejsza informacja jest na samym końcu):
### `Input should be 'Intern', 'Junior' ... [type=literal_error, input_value='Senior Manager']`

#### Co się stało?
1.  Model przeczytał "Senior Manager".
2.  Stwierdził: "Ok, to jest Senior Manager".
3.  Biblioteka `Pydantic` sprawdza: "Czy 'Senior Manager' jest na liście `Literal`?".
4.  Nie jest.
5.  Rzuciła wyjątkiem.

### Jak to interpretować?
Możesz myśleć o `Pydantic` nie tylko jako o sposobie definiowania schematu. To jest jedna funkcja, a druga – którą już widzisz – to bramka jakości.

To znaczy, jeśli nam na czymś zależy, możemy ustawiać i uszczelniać różnego rodzaju walidacje. Jeśli dana wartość przeszła przez bramkę jakości, oznacza to, że nasze walidatory zostały spełnione. To bardzo dobra wiadomość – jeśli uszczelniamy najbardziej krytyczne rzeczy, na których nam zależy, i nie pojawia się wyjątek, oznacza to, że system jest stabilny.

Pamiętaj, że LLM nie może Ci gwarantować stabilności. To oznacza, że potrzebujesz przenieść elementy stabilności poza LLM. W tym przypadku stabilność zapewnia `Pydantic`, czyli zwykłe oprogramowanie (Software 1.0). Łączymy więc Software 3.0, który ma swoje talenty, ale problemy ze stabilnością, z Software 1.0, który jest bardziej jednoznaczny – pozwala sprawdzać dokładnie to, na czym Ci zależy.

Gdybyśmy nie mieli tej walidacji, do Twojej bazy trafiłby "śmieć" w postaci np. `Senior Manager`, który popsułby późniejsze raporty (bo system zliczałby osobno `Manager` i osobno `Senior Manager` lub jeszcze inna badziej dziwniejsza kombinacja).

### Jak to naprawić? 

#### Strategia 1: Pętla naprawcza (`max_retries`)

To pierwsza linia obrony. Zwróć uwagę na parametr `max_retries` w naszej funkcji `call_api`.

```python
# ...
max_retries = 2
# ...
```

Jak to działa pod maską? To nie jest magia, to czysta inżynieria biblioteki `instructor`.
Gdy ustawisz `max_retries=2`, dzieje się następująca sekwencja:

1.  **Próba 1:** Wysyłasz zapytanie. Model zwraca `"Senior Manager"`.
2.  **Walidacja:** Pydantic krzyczy: *"Błąd! To nie jest dozwolona wartość!"*.
3.  **Interwencja (To jest klucz!):** Biblioteka `instructor` przechwytuje ten błąd. Ale zamiast wywalić program, robi coś sprytnego:
    *   Bierze treść błędu (ten czerwony tekst, który widziałeś wyżej).
    *   **Dokleja go do historii rozmowy** jako nową wiadomość od użytkownika.
    *   Wysyła zapytanie ponownie (Próba 2).
4.  **Próba 2:** Model widzi teraz: *"Hej, użytkownik pytał o X, ja odpowiedziałem 'Senior Manager', ale dostałem błąd walidacji: Input should be 'Intern', 'Junior'... Popraw się!"*.
5.  **Sukces (często):** Model "rozumie" swój błąd i za drugim razem zwraca poprawną wartość (np. "Manager").




In [ ]:
call_api(
    [{"role": "user", "content": "Senior Manager, Business Intelligence, InTech"}], 
    JobTitleV3,
    "speakleash/Bielik-11B-v3.0-Instruct",
    max_retries=2
) 

## Mamy to `JobTitleV3(seniority_level='Senior')`


Widzisz, że za drugim razem model zrozumiał, czego od niego chcemy, i dopasował się. Poniżej widać sekwencję procesu naprawczego:
Najpierw jest pierwsza próba – jeśli się nie uda, pojawia się błąd i wykonuje się druga próba.

Fizycznie wykonujemy wtedy dwa wywołania API (więcej płacimy, dłużej czekamy itd).

```ascii

┌────────────────────────────────────────────────────────────────────────────┐
│                   JAK DZIAŁA max_retries = 2 W INSTRUCTOR                  │
│                                                                            │
│   Model:  JobTitleV3                                                       │
│   Pole:   seniority_level: Optional[SeniorityLevel]                        │
│   Dozwolone: Intern │ Junior │ Mid-Level │ Senior │ Lead │ Staff │         │
│              Principal │ Manager │ Director │ Executive                    │
└────────────────────────────────────────────────────────────────────────────┘

  Twój kod                    Instructor                   OpenAI API
     │                            │                            │
     │  client.chat.completions   │                            │
     │  .create(max_retries=2)    │                            │
     │ ──────────────────────────>│                            │
     │                            │                            │
     │                            │  ┌───────────────────────┐ │
     │                            │  │ 📡 PRÓBA 1/2          │ │
     │                            │  └───────────────────────┘ │
     │                            │                            │
     │                            │  messages = [              │
     │                            │    {"role": "user",        │
     │                            │     "content":             │
     │                            │     "Wyekstrahuj dane      │
     │                            │      ze stanowiska:        │
     │                            │      Senior IT Manager"}   │
     │                            │  ]                         │
     │                            │ ──────────────────────────>│
     │                            │                            │
     │                            │  {"seniority_level":       │
     │                            │   "Senior Manager"}    ❌  │
     │                            │ <──────────────────────────│
     │                            │                            │
     │                    ┌───────┴────────┐                   │
     │                    │  🔍 WALIDACJA  │                   │
     │                    │    Pydantic    │                   │
     │                    └───────┬────────┘                   │
     │                            │                            │
     │              ╔═════════════╧══════════════╗             │
     │              ║  ❌ ValidationError        ║             │
     │              ║                            ║             │
     │              ║  1 validation error for    ║             │
     │              ║  JobTitleV3                ║             │
     │              ║  seniority_level           ║             │
     │              ║    Input should be         ║             │
     │              ║    'Intern', 'Junior',     ║             │
     │              ║    'Mid-Level', 'Senior',  ║             │
     │              ║    'Lead', 'Staff',        ║             │
     │              ║    'Principal', 'Manager', ║             │
     │              ║    'Director' or           ║             │
     │              ║    'Executive'             ║             │
     │              ║                            ║             │
     │              ║  input_value='Senior       ║             │
     │              ║  Manager'                  ║             │
     │              ╚═════════════╤══════════════╝             │
     │                            │                            │
     │              ┌─────────────┴──────────────┐             │
     │              │  🔧 INSTRUCTOR             │             │
     │              │                            │             │
     │              │  1. Przechwytuje błąd      │             │
     │              │  2. Bierze odpowiedź LLM   │             │
     │              │     → dodaje jako assistant│             │
     │              │  3. Bierze treść błędu     │             │
     │              │     → dodaje jako user     │             │
     │              │  4. Wysyła CAŁOŚĆ ponownie │             │
     │              └─────────────┬──────────────┘             │
     │                            │                            │
     │                            │  ┌───────────────────────┐ │
     │                            │  │ 📡 PRÓBA 2/2          │ │
     │                            │  └───────────────────────┘ │
     │                            │                            │
     │                            │  messages = [              │
     │                            │                            │
     │                            │  ┌─ msg 1 ──────────────┐  │
     │                            │  │ role: "user"         │  │
     │                            │  │                      │  │
     │                            │  │ "Wyekstrahuj dane    │  │
     │                            │  │  ze stanowiska:      │  │
     │                            │  │  Senior IT Manager"  │  │
     │                            │  └──────────────────────┘  │
     │                            │                            │
     │                            │  ┌─ msg 2 ──────────────┐  │
     │                            │  │ role: "assistant"    │  │
     │                            │  │                      │  │
     │                            │  │ {"seniority_level":  │  │
     │                            │  │  "Senior Manager"}   │  │
     │                            │  └──────────────────────┘  │
     │                            │                            │
     │                            │  ┌─ msg 3 ──────────────┐  │
     │                            │  │ role: "user"         │  │
     │                            │  │                      │  │
     │                            │  │ "Recall the function │  │
     │                            │  │  correctly, fix the  │  │
     │                            │  │  errors:             │  │
     │                            │  │                      │  │
     │                            │  │  1 validation error  │  │
     │                            │  │  for JobTitleV3      │  │
     │                            │  │  seniority_level     │  │
     │                            │  │   Input should be    │  │
     │                            │  │   'Intern','Junior', │  │
     │                            │  │   'Mid-Level',       │  │
     │                            │  │   'Senior','Lead',   │  │
     │                            │  │   'Staff',           │  │
     │                            │  │   'Principal',       │  │
     │                            │  │   'Manager',         │  │
     │                            │  │   'Director' or      │  │
     │                            │  │   'Executive'        │  │
     │                            │  │                      │  │
     │                            │  │  input_value=        │  │
     │                            │  │  'Senior Manager'"   │  │
     │                            │  └──────────────────────┘  │
     │                            │                            │
     │                            │  ]                         │
     │                            │ ──────────────────────────>│
     │                            │                            │
     │                            │  {"seniority_level":       │
     │                            │   "Senior"}            ✅  │
     │                            │ <──────────────────────────│
     │                            │                            │
     │                    ┌───────┴────────┐                   │
     │                    │  🔍 WALIDACJA  │                    │
     │                    │    Pydantic    │                   │
     │                    └───────┬────────┘                   │
     │                            │                            │
     │              ╔═════════════╧═══════════════╗            │
     │              ║  ✅ SUKCES!                 ║            │
     │              ║                             ║            │
     │              ║  "Senior" ∈  SeniorityLevel ║            │
     │              ║  Walidacja przeszła ✓       ║            │
     │              ╚═════════════╤═══════════════╝            │
     │                            │                            │
     │  JobTitleV3(               │                            │
     │    seniority_level=        │                            │
     │      "Manager"             │                            │
     │  )                         │                            │
     │ <──────────────────────────│                            │
     │                            │                            │
     ▼                            ▼                            ▼


┌─────────────────────────────────────────────────────────────────────────────┐
│                                                                             │
│  CO MODEL "WIDZI" W PRÓBIE 2:                                               │
│                                                                             │
│  ┌────────────────────────────────────────────────────────────────────┐     │
│  │  👤 User:      "Wyekstrahuj dane: Senior IT Manager"               │     │
│  │  🤖 Assistant: {"seniority_level": "Senior Manager"}               │     │
│  │  👤 User:      "Popraw się! 'Senior Manager' nie jest dozwolone.   │     │
│  │                 Wybierz z: Intern, Junior, Mid-Level, Senior,      │     │
│  │                 Lead, Staff, Principal, Manager, Director,         │     │
│  │                 Executive"                                         │     │
│  └────────────────────────────────────────────────────────────────────┘     │
│                                                                             │
│  Model "rozumie":                                                           │
│  "Senior Manager" ≠ żadna z opcji → najbliższe dopasowanie → "Senior "      │
│                                                                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  A GDYBY max_retries = 1?                                                   │
│                                                                             │
│  Próba 1 → "Senior Manager" → ❌ ValidationError                            │
│  Brak kolejnej próby → 💥 InstructorRetryException                          │
│                                                                             │
│  InstructorRetryException: 1 validation error for JobTitleV3                │
│  seniority_level                                                            │
│    Input should be 'Intern', 'Junior', 'Mid-Level', 'Senior',               │
│    'Lead', 'Staff', 'Principal', 'Manager', 'Director' or 'Executive'       │
│    [type=literal_error, input_value='Senior Manager',                       │
│     input_type=str]                                                         │
│                                                                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  PODSUMOWANIE:                                                              │
│                                                                             │
│  max_retries = 1  →  1 wywołanie API  →  0 szans na poprawę                 │
│  max_retries = 2  →  do 2 wywołań API →  1 szansa na poprawę                │
│  max_retries = 3  →  do 3 wywołań API →  2 szanse na poprawę                │
│                                                                             │
│  ⚠️  Każdy retry = dodatkowy API call = dodatkowy koszt + czas               │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```


To działa świetnie przy dużych modelach, ale przy mniejszych model czasem bywa uparty. Nawet jak dostanie błąd, może znowu zwrócić to samo, albo zacząć przepraszać ("Przepraszam, już poprawiam..."), co znowu rozwali format JSON.

Dlatego czasem możemy potrzebuować cięższych dział.

### Strategia 2: Walidator z logiką naprawczą (+naturalna inteligencja)

Skoro sztuczna inteligencja sobie nie radzi (lub jest niedeterministyczna), możemy to skompensować naszą naturalną inteligencją.

Możemy z góry przewidzieć, gdzie model będzie się mylił. "Senior Manager" to klasyczny przykład. Zamiast liczyć na to, że model się domyśli w drugiej próbie (co kosztuje czas i tokeny), możemy wstrzyknąć **logikę naprawczą** bezpośrednio w kod.

Użyjemy dekoratora `@model_validator(mode='before')`.
To taki "bramkarz", który stoi przed modelem Pydantic
1.  Przechwytuje "surowy" wynik od modelu (ten błędny)
2.  Uruchamia Twój kod Pythonowy (Twoje reguły if/else)
3.  Naprawia dane "w locie"
4.  Dopiero tak naprawione dane wpuszcza do właściwej walidacji

### Dlaczego to może mieć sens?
*   **Kontrola:** Masz 100% pewności, jak zostanie obsłużony dany przypadek.
*   **Koszt:** Nie wykonujesz drugiego zapytania do API (oszczędzasz czas).
*   **Czystość:** Twój słownik `Literal` pozostaje czysty (nie musisz dodawać do niego 50 dziwnych wariantów typu "Snr Mngr").

To jest hybryda:
Zobaczmy, jak to napisać (będzie więcej kodu, ale poniżej wyjaśnię go dokładniej) 👇

In [ ]:
class JobTitleV4(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None, 
        description="Poziom doświadczenia wymagany na stanowisku"
    )

    @model_validator(mode='before')
    @classmethod
    def normalize_seniority(cls, data: Any) -> Any:
        # Sprawdzamy czy wejście to słownik (typowe dla Pydantic)
        if isinstance(data, dict):
            raw_value = data.get('seniority_level')
            
            # Jeśli wartość istnieje i jest stringiem, próbujemy ją naprawić
            if raw_value and isinstance(raw_value, str):
                # Normalizacja tekstu (np. usunięcie spacji, wielkie litery)
                text = raw_value.strip().title()
                
                # Lista dozwolonych wartości dla szybkiego sprawdzenia
                # Używamy get_args, aby pobrać wartości z Literal dynamicznie
                from typing import get_args
                allowed_values = set(get_args(SeniorityLevel))
                
                # 1. Jeśli model trafił idealnie – nie robimy nic
                if text in allowed_values:
                    return data

                # 2. Logika naprawcza (Heurystyka)
                # Kolejność warunków jest krytyczna!
                # "Senior Manager" -> zawiera "Manager", więc wpadnie w Manager (ważniejsze niż Senior)
                # "Staff Senior Engineer" -> zawiera "Staff", więc wpadnie w Staff
                
                mapped_value = None

                if any(x in text for x in ["C-Level", "Vp", "President", "Chief", "Executive"]):
                    mapped_value = "Executive"
                elif any(x in text for x in ["Director", "Head"]):
                    mapped_value = "Director"
                elif "Manager" in text:
                    mapped_value = "Manager"  # Tutaj wpada "Senior Manager"
                elif any(x in text for x in ["Principal", "Distinguished"]):
                    mapped_value = "Principal"
                elif "Staff" in text:
                    mapped_value = "Staff"
                elif "Lead" in text:
                    mapped_value = "Lead"
                elif "Senior" in text:
                    mapped_value = "Senior"
                elif any(x in text for x in ["Junior", "Associate", "Entry"]):
                    mapped_value = "Junior"
                elif any(x in text for x in ["Intern", "Student", "Trainee"]):
                    mapped_value = "Intern"
                elif "Mid" in text or "Regular" in text:
                    mapped_value = "Mid-Level"
                
                # Jeśli udało się zmapować, podmieniamy wartość w słowniku
                if mapped_value:
                    data['seniority_level'] = mapped_value
                
                # Opcjonalnie: Jeśli nic nie pasuje, można ustawić None lub "Mid-Level"
                # data['seniority_level'] = None 
                
        return data

## Za dużo kodu?
Ten kod może wyglądać na skomplikowany (trochę "dużo literek"), ale jego logika jest prosta, gdy podzielimy go na kawałki.

To jest Twoja instrukcja bezpieczeństwa. Zamiast liczyć na to, że model "może się domyśli" za drugim razem (co kosztuje czas), my sztywno kodujemy reguły naprawcze. Zwykle nie chcemy tego robić, ale gdy nie mamy wyjścia (bo możemy używać tylko lokalnego modelu), pojawia się motywacja, żeby sobie z tym poradzić.

Podzielmy to na 4 logiczne grupy:

### 1. Grupa "bramkarz" (przechwycenie danych)
```python
@model_validator(mode='before')
@classmethod
def normalize_seniority(cls, data: Any) -> Any:
```
To jest najważniejszy moment. Dekorator `@model_validator(mode='before')` działa jak bramkarz przed wejściem do klubu.
Zamiast wpuszczać dane prosto do walidacji (gdzie Pydantic wyrzuciłby błąd na widok "Senior Manager"), my je **przechwytujemy**.
Mamy szansę "poprawić krawat" (dane), zanim "finalny" bramkarz sprawdzi, czy pasują do dress code'u.

### 2. Grupa "higiena" (normalizacja)
```python
text = raw_value.strip().title()
allowed_values = set(get_args(SeniorityLevel))

if text in allowed_values:
    return data
```
Tutaj robimy podstawowe porządki:
1.  Usuwamy spacje (`strip`) i poprawiamy wielkie litery (`title`). Dzięki temu `"   junior "` zamieni się w `"Junior"`.
2.  **Sprawdzamy "happy path":** Jeśli model zwrócił idealną wartość (np. "Junior"), która jest w naszym słowniku -> **NIC NIE ROBIMY**. Zwracamy dane i kończymy pracę. Nie ma sensu poprawiać czegoś, co jest dobre, prawda?

### 3. Grupa "Warsztat naprawczy" (heurystyka) - to tutaj MODYFIKUJESZ rzeczy i naprawiasz świat
To jest serce tego kodu. Tutaj naturalna Inteligencja naprawia błędy sztucznej inteligencji.
Stosujemy prostą logikę "jeśli zawiera słowo X, to przypisz kategorię Y".

**⚠️ UWAGA: Kolejność ma znaczenie krytyczne!**

```python
elif "Manager" in text:
    mapped_value = "Manager"
# ... (później)
elif "Senior" in text:
    mapped_value = "Senior"
```

Dlaczego taka kolejność?
Wyobraź sobie tytuł: **"Senior Manager"**.
- Gdybyśmy najpierw sprawdzali słowo "Senior", kod przypisałby kategorię `Senior`. **BŁĄD.** To jest rola zarządcza.
- Dlatego najpierw sprawdzamy "Manager". Jeśli kod znajdzie to słowo, przypisuje `Manager` i kończy sprawdzanie.

**Co tu zmieniasz?**
Jeśli jutro zauważysz, że model często myli "Tech Lead" z "Managerem", to tutaj dopisujesz nowy `elif` z Twoją regułą. To jest Twoja tablica bezpieczników.

### 4. Grupa "aplikacja poprawki"
```python
if mapped_value:
    data['seniority_level'] = mapped_value
return data
```
Jeśli nasza logika coś znalazła (`mapped_value`), podmieniamy brudną wartość (np. "Senior Manager") na czystą (np. "Manager") w słowniku danych.
Dopiero tak naprawiony słownik zwracamy do Pydantic.

Teraz Pydantic patrzy na to i mówi: *"O, dostałem 'Manager'. To jest na liście dozwolonych wartości. Wpuszczam!"*.
Błąd zniknął, system działa dalej.

### Sprawdźmy to w praktyce

Teraz "Senior Manager" powinien przejść gładko jako "Manager".

In [ ]:
call_api(
    [{"role": "user", "content": "Senior Manager, Business Intelligence, InTech"}], 
    JobTitleV4,
    "speakleash/Bielik-11B-v3.0-Instruct",
    max_retries=1
) 

## Jak widzisz, działa 🥳

`max_retries=1` jest ustawione na 1, czyli był tylko jeden call do API modelu.
Ale... wynik to `Manager`.

Dlaczego?


```python
    
elif "Manager" in text:
   mapped_value = "Manager"  # Tutaj wpada "Senior Manager"

```

Zawsze możesz to zmienić i obsłużyć inaczej. Na razie jednak bardziej poznajemy, jakie są opcje i jak można sobie z tym poradzić.

## Testy przypadków brzegowych

Co równie ważne i kluczowe z punktu widzenia stabilności: możesz pisać testy. Dokładnie takie same testy, które pilnowały jakości w Software 1.0.

Dzięki temu, nawet jeśli przegapisz jakiś przypadek, dodajesz go jako kolejny test i masz pewność, że wszystko, co było wcześniej, plus ten nowy przypadek, będzie pilnowane przez bramkę jakości. Tak — to wymaga czasu na kalibrację i stabilizację. Nie zawsze da się zdefiniować wszystko z góry, ale jest to naturalnie wpisane w czas realizacji projektu jako faza stabilizacji.

Sprawdźmy teraz, czy nasza logika działa dla różnych wariantów.

In [ ]:
# Przypadek 1: Model zwraca "Senior Manager" (Błąd w V3, Sukces w V4)
job1 = JobTitleV4(seniority_level="Senior Manager")
print(f"Input: 'Senior Manager' -> Output: {job1.seniority_level}") 
# Output: Manager

# Przypadek 2: Model zwraca "Senior Backend Developer"
job2 = JobTitleV4(seniority_level="Senior Backend Developer")
print(f"Input: 'Senior Backend Developer' -> Output: {job2.seniority_level}")
# Output: Senior

# Przypadek 3: Model zwraca "Team Lead"
job3 = JobTitleV4(seniority_level="Team Lead")
print(f"Input: 'Team Lead' -> Output: {job3.seniority_level}")
# Output: Lead

# Przypadek 4: Poprawna wartość
job4 = JobTitleV4(seniority_level="Junior")
print(f"Input: 'Junior' -> Output: {job4.seniority_level}")

A teraz czas podziałać samodzielnie.


## 🧠 Zadanie domowe (Dzień 2)

Gratuluję — właśnie już umiesz łączyć "nieokreśloność" LLM-a ze "sztywnością" inżynierii danych.
Potrafisz wyciągnąć `seniority_level` z dowolnego tytułu stanowiska.

Ale biznes potrzebuje więcej.

Wyobraź sobie, że HR pyta: *"OK, wiemy że to Senior. Ale czy ta osoba zarządza ludźmi? Czy to rola badawcza? Czy to kontraktor?"*

**Twoim zadaniem** jest rozbudowanie modelu `JobTitleV5` o dodatkowe pola:

1. `normalized_title` — Znormalizowany tytuł (np. `"Data Scientist"` zamiast *"Senior Data Scientist IV"*)
2. `is_management` — Czy to rola menedżerska? (`True`/`False`)
3. `is_research` — Czy to rola badawcza?
4. `is_consulting` — Czy to rola "dla klienta"?
5. `is_operations` — Czy to DevOps/MLOps?
6. `is_contractor` — Czy to B2B/Freelance?

Przetestuj swój model na tych przykładach:

```python
test_titles = [
    "Senior Machine Learning Engineer",           # is_management=False, is_operations=False
    "VP of Data Science",                          # is_management=True
    "Contract Data Analyst - 6 months",            # is_contractor=True
    "MLOps Lead",                                  # is_operations=True
    "Research Scientist, NLP (Ph.D. required)",    # is_research=True
    "Solutions Architect - Financial Services",    # is_consulting=True
]
```

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈
</summary>
<p>
Poniżej masz szkielet. Twoim zadaniem jest wypełnić opisy w <code>description="..."</code>.
</p>
<br>
Pamiętaj zasadę, że <strong>opis pola to prompt</strong>? Im precyzyjniej opiszesz, <em>kiedy</em> <code>is_management</code> ma być <code>True</code> (a kiedy nie!), tym lepiej model zadziała.

```python
NormalizedTitle = Literal[
    "Data Scientist", "Data Engineer", "Data Analyst",
    "Machine Learning Engineer", "MLOps Engineer",
    "Research Scientist", "Software Engineer",
    "DevOps Engineer", "Solutions Architect",
    "Product Manager", "Engineering Manager",
    # ... dodaj brakujące role
]

class JobTitleV5(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description="..."
    )
    normalized_title: Optional[NormalizedTitle] = Field(
        None,
        description="..."
    )
    is_management: bool = Field(False, description="...")
    is_research:   bool = Field(False, description="...")
    is_consulting: bool = Field(False, description="...")
    is_operations: bool = Field(False, description="...")
    is_contractor: bool = Field(False, description="...")
```

<p>
<strong>Największa pułapka:</strong> Jak odróżnić <code>is_management=True</code> od Tech Leada?
<br>
Tech Lead to lider techniczny — <strong>nie zarządza ludźmi</strong> (no HR responsibility).
<br>
Manager/Director/VP — <strong>zarządza ludźmi</strong> (hiring, firing, reviews).
<br>
Właśnie takie niuanse musisz wpisać w <code>description</code>.
</p>

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈
</summary>

<p>

Tu masz kompletne rozwiązanie. Zwróć uwagę, jak precyzyjne są opisy — to one robią całą robotę:
</p>

```python
NormalizedTitle = Literal[
    # Czyste role — bez seniorów, bez dyrektorów
    "Data Scientist", "Data Engineer", "Data Analyst",
    "Analytics Engineer", "BI Developer",
    "Data Architect", "Solutions Architect",
    "Machine Learning Engineer", "MLOps Engineer",
    "AI Engineer", "Computer Vision Engineer", "NLP Engineer",
    "Research Scientist", "Applied Scientist",
    "Software Engineer", "Backend Engineer", "Fullstack Engineer",
    "DevOps Engineer", "Platform Engineer", "SRE",
    "Product Manager", "Product Owner",
    "Engineering Manager",
]


class JobTitleV5(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description="Poziom doświadczenia wymagany na stanowisku"
    )

    normalized_title: Optional[NormalizedTitle] = Field(
        None,
        description=(
            "Znormalizowana nazwa stanowiska. "
            "Ignoruj poziom seniority (np. Senior, Junior) "
            "oraz lokalizację."
        )
    )

    is_management: bool = Field(
        False,
        description="""
        Czy stanowisko obejmuje bezpośrednie zarządzanie
        ludźmi (people management)?
        True dla: Manager, Director, VP, Head of.
        False dla: Lead, Staff, Principal
        — to liderzy techniczni BEZ HR responsibility.
        """
    )

    is_research: bool = Field(
        False,
        description="""
        Czy stanowisko ma charakter badawczy?
        True dla: Research Scientist, Research Engineer,
        Applied Scientist.
        Typowo wymaga: publikacje, Ph.D.
        False dla: ML Engineer (produkcja), Data Analyst.
        """
    )

    is_consulting: bool = Field(
        False,
        description="""
        Czy rola jest konsultingowa / client-facing?
        True dla: Consultant, Solutions Architect,
        Pre-Sales, Customer Success.
        """
    )

    is_operations: bool = Field(
        False,
        description="""
        Czy rola jest operacyjna (infra/production)?
        True dla: MLOps, DevOps, DataOps, SRE,
        Platform Engineer.
        Focus: deployment, monitoring, CI/CD.
        """
    )

    is_contractor: bool = Field(
        False,
        description="""
        Czy stanowisko jest kontraktowe/tymczasowe?
        True dla: Contract, Contractor, Freelance,
        Fixed-Term (FTC), B2B.
        """
    )


################ TEST


test_titles = [
    "Senior Machine Learning Engineer",           # is_management=False, is_operations=False
    "VP of Data Science",                          # is_management=True
    "Contract Data Analyst - 6 months",            # is_contractor=True
    "MLOps Lead",                                  # is_operations=True
    "Research Scientist, NLP (Ph.D. required)",    # is_research=True
    "Solutions Architect - Financial Services",    # is_consulting=True
]



for title in test_titles:
    print(title)
    response = call_api(
        [{"role": "user", "content": title}], 
        JobTitleV5,
        "speakleash/Bielik-11B-v3.0-Instruct",
        max_retries=2
    ) 
    print(response)
    print("=" * 100)    
```

### Dlaczego to działa?

Porównaj dwa podejścia:

```
❌  is_management: bool = Field(False, description="Management flag")
✅  is_management: bool = Field(False, description="True dla: Manager,
    Director, VP. False dla: Lead, Staff — to liderzy techniczni
    BEZ HR responsibility.")
```

Pierwsza wersja zmusza model do zgadywania — *"Czy Tech Lead to management?"*.
Druga **explicite** mówi: *"Lead = NIE"*. Zero dwuznaczności.

To jest sedno **Software 3.0**:  piszesz precyzyjne opisy, a model sam buduje klasyfikator.

</details>
</details>

In [ ]:
## YOUR CODE HERE

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie dnia 2</h2>
<h4>Co masz teraz zrobić?</h4>

<ul>
    <ol>📸 <b>Zrób Screenshot:</b> Zrób zrzut ekranu wyniku powyższej pętli.</ol>
    <ol>💬 <b>Wrzuć na Slacka:</b> kanał <a href="https://dataworkshopclub.slack.com/archives/C0ADAKBRX3N">`#dwthon_llm_day2_done`</a>.</ol>
</ul>
</div>

### Co dalej? (dzień 3)

Masz już potężną maszynę do strukturyzowania tytułów. Ale tytuł to tylko nagłówek.
Prawdziwe "mięso" (i prawdziwy bałagan) siedzi w **opisie ogłoszenia (Description)**.

Tam są ściany tekstu, listy wymagań, opisy benefitów (wiadomo "owocowe czwartki" i pewnie coś jeszcze) i technologie ukryte między wierszami.
Jutro nauczymy się, jak wyciągać z tego skomplikowanego tekstu konkretne **technologie (Skills)** oraz **widełki płacowe (Salary)**, nawet jeśli są zapisane jako "10k - 15k PLN netto B2B".


### Podsumowanie

To był intensywny dzień, dokładnie tak jak obiecywałem. Zostawiliśmy za sobą teoretyczne rozważania i weszliśmy w samo serce inżynierii danych.

Zaczęliśmy od tytułów, gdzie każdy rekruter (lub ktoś inny) wpisuje to, co mu przyjdzie do głowy. Mogliśmy zostać przy prostym wyciąganiu jako tekst (i tam może być cokolwiek), ale w praktyce biznesowej to droga donikąd. Dlatego narzuciliśmy modelowi żelazną strukturę.

Co faktycznie osiągnęliśmy?

Przełamaliśmy barierę między "nieomylnym kodem" a "halucynującym modelem". Wiesz już, że model LLM bez odpowiedniego kagańca w postaci typów `Literal` i walidacji `Pydantic` jest zwykle bezużyteczny w procesach automatyzacji. Ale co ważniejsze – nauczyliśmy się obsługiwać sytuacje, gdy ten kaganiec okazuje się zbyt ciasny.

Masz teraz w ręku dwie ciekawe strategie: automatyczną naprawę przez `max_retries` (gdy liczy się czas dewelopera) oraz własną logikę naprawczą w `model_validator` (gdy liczy się koszt tokenów i precyzja). To jest ten moment, w którym przestajesz być tylko użytkownikiem AI, a stajesz się praktykiem budującym odporne systemy Software 3.0.

Jutro czeka nas kolejne wyzwanie. Skoro tytuły mamy uporządkowane, weźmiemy się za "mięso" – opisy ogłoszeń pełne treści, formatowania i "innych rzeczy". Do jutra!